In [ ]:
# =========================================================
# C3 DIFFERENTIATION PIPELINE
# Serper + OpenRouter(OpenAI SDK) + Python Scoring
# =========================================================

# INSTALL IF NEEDED:
# !pip install openai pandas requests openpyxl

import pandas as pd
import requests
import json
import time
from openai import OpenAI

# =========================================================
# CONFIG
# =========================================================

OPENROUTER_API_KEY=your_key_here
SERPER_API_KEY=your_key_here

MODEL_NAME = "openai/gpt-4o-mini"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

# =========================================================
# LOAD DATA
# =========================================================

# companies.xlsx should contain:
# Company Name | Website

df = pd.read_excel("companies.xlsx")

# =========================================================
# SERPER SEARCH FUNCTION
# =========================================================

def serper_search(company_name):

    url = "https://google.serper.dev/search"

    query = f"""
    {company_name}
    patents DSIR USFDA EU-GMP WHO-GMP
    proprietary products custom synthesis
    specialty chemicals innovation R&D pioneer
    """

    payload = {
        "q": query,
        "num": 8
    }

    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    try:

        response = requests.post(
            url,
            headers=headers,
            json=payload,
            timeout=30
        )

        if response.status_code != 200:

            print(f"Serper Error {response.status_code} for {company_name}")

            return ""

        data = response.json()

        snippets = []

        for item in data.get("organic", [])[:8]:

            title = item.get("title", "")
            snippet = item.get("snippet", "")
            link = item.get("link", "")

            combined = f"""
TITLE: {title}
SNIPPET: {snippet}
LINK: {link}
"""

            snippets.append(combined)

        return "\n".join(snippets)

    except Exception as e:

        print(f"Serper Exception for {company_name}: {e}")

        return ""

# =========================================================
# OPENAI DIFFERENTIATION DETECTION
# =========================================================

def detect_differentiation_ai(company_name, search_text):

    prompt = f"""
You are evaluating whether a manufacturing company is technically differentiated.

Company:
{company_name}

Search Evidence:
{search_text}

Evaluate evidence for these signals:

1. Patents
2. DSIR Recognition
3. Regulatory Approvals
(USFDA, EU-GMP, WHO-GMP, certified facilities)

4. Proprietary Products/Processes
5. Pioneer/First Claims
6. Custom Synthesis/Niche Chemistry
7. Commodity/Basic Chemicals

Classify each signal as:
- Strong
- Weak
- Absent

IMPORTANT:
- Use only provided evidence
- Weak evidence is allowed
- Do NOT hallucinate
- Return ONLY valid JSON

Return STRICT JSON ONLY:

{{
  "Patents": "Strong/Weak/Absent",
  "DSIR": "Strong/Weak/Absent",
  "Regulatory Approvals": "Strong/Weak/Absent",
  "Proprietary Products": "Strong/Weak/Absent",
  "Pioneer Claims": "Strong/Weak/Absent",
  "Custom Synthesis": "Strong/Weak/Absent",
  "Commodity Chemicals": "Strong/Weak/Absent",
  "reason": ""
}}
"""

    try:

        response = client.chat.completions.create(

            model=MODEL_NAME,

            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a strict manufacturing research analyst. "
                        "Return only valid JSON."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=0,
            max_tokens=500
        )

        text = response.choices[0].message.content.strip()

        # Remove markdown formatting
        text = text.replace("```json", "")
        text = text.replace("```", "")
        text = text.strip()

        # Robust JSON parsing
        try:

            result = json.loads(text)

        except:

            start = text.find("{")
            end = text.rfind("}") + 1

            cleaned = text[start:end]

            result = json.loads(cleaned)

        return result

    except Exception as e:

        print(f"OpenAI Error for {company_name}: {e}")

        return {
            "Patents": "Absent",
            "DSIR": "Absent",
            "Regulatory Approvals": "Absent",
            "Proprietary Products": "Absent",
            "Pioneer Claims": "Absent",
            "Custom Synthesis": "Absent",
            "Commodity Chemicals": "Absent",
            "reason": f"API Error: {str(e)}"
        }

# =========================================================
# C3 WEIGHTED SCORING
# =========================================================

def calculate_c3_score(result):

    signal_weights = {

        "Patents": 2,
        "DSIR": 2,
        "Regulatory Approvals": 2,
        "Proprietary Products": 1,
        "Pioneer Claims": 1,
        "Custom Synthesis": 1,
        "Commodity Chemicals": -2
    }

    detected_signals = []

    total_weight = 0

    for signal, weight in signal_weights.items():

        value = str(result.get(signal, "Absent")).strip().title()

        if value == "Strong":

            total_weight += weight

            detected_signals.append(
                f"{signal} (Strong)"
            )

        elif value == "Weak":

            total_weight += weight * 0.5

            detected_signals.append(
                f"{signal} (Weak)"
            )

    # FINAL C3 SCORE

    if total_weight >= 4:
        c3_score = 25

    elif total_weight >= 1:
        c3_score = 12

    else:
        c3_score = 0

    return (
        detected_signals,
        total_weight,
        c3_score
    )

# =========================================================
# CONFIDENCE FUNCTION
# =========================================================

def get_confidence(total_weight):

    if total_weight >= 4:
        return "High"

    elif total_weight >= 1:
        return "Medium"

    return "Low"

# =========================================================
# MAIN PIPELINE
# =========================================================

results = []

total_companies = len(df)

for idx, row in df.iterrows():

    company = str(row["Company Name"]).strip()

    website = ""

    if "Website" in df.columns:
        website = str(row["Website"]).strip()

    print(f"\n[{idx+1}/{total_companies}] Processing: {company}")

    # -----------------------------------------------------
    # STEP 1: SERPER SEARCH
    # -----------------------------------------------------

    search_text = serper_search(company)

    print(search_text[:400])

    # -----------------------------------------------------
    # HANDLE EMPTY SEARCH RESULTS
    # -----------------------------------------------------

    if len(search_text.strip()) < 50:

        results.append({
            "Company Name": company,
            "Website": website,
            "Differentiation Signals": "",
            "Signal Weight": 0,
            "C3 Score": 0,
            "Confidence": "Low",
            "Reason": "No search evidence retrieved",
            "Search Evidence": ""
        })

        continue

    # -----------------------------------------------------
    # STEP 2: AI DIFFERENTIATION DETECTION
    # -----------------------------------------------------

    ai_result = detect_differentiation_ai(
        company,
        search_text[:2500]
    )

    # -----------------------------------------------------
    # STEP 3: PYTHON SCORING
    # -----------------------------------------------------

    (
        detected_signals,
        total_weight,
        c3_score
    ) = calculate_c3_score(ai_result)

    confidence = get_confidence(total_weight)

    # -----------------------------------------------------
    # STORE RESULTS
    # -----------------------------------------------------

    results.append({

        "Company Name": company,

        "Website": website,

        "Differentiation Signals":
            ", ".join(detected_signals),

        "Signal Weight":
            total_weight,

        "C3 Score":
            c3_score,

        "Confidence":
            confidence,

        "Reason":
            ai_result.get("reason", ""),

        "Search Evidence":
            search_text[:2500]
    })

    # -----------------------------------------------------
    # RATE LIMIT PROTECTION
    # -----------------------------------------------------

    time.sleep(6)

# =========================================================
# SAVE OUTPUT
# =========================================================

output_df = pd.DataFrame(results)

output_df.to_excel(
    "c3_differentiation_scores.xlsx",
    index=False
)

print("\n===================================")
print("DONE")
print("Saved: c3_differentiation_scores.xlsx")
print("===================================")


[1/18] Processing: A R Life Sciences

TITLE: AR Life Sciences Private Limited - PharmaCompass.com
SNIPPET: AR Life Sciences Private Limited// We are into manufacturing of bulk drug Intermediates and Active Pharmaceutical Ingredients (API).
LINK: https://www.pharmacompass.com/api-manufacturers/ar-life-sciences-private-limited


TITLE: [PDF] PHỤ LỤC 2: DANH SÁCH CƠ SỞ SẢN XUẤT THUỐC, NGUYÊN ...
SNIPPET: Xuất xưởng lô thuốc vô trùng. * 

[2/18] Processing: lucent drugs

TITLE: [PDF] Patent protection as a key driver for pharmaceutical innovation | IFPMA
SNIPPET: A robust, time-limited system of patent protection is proven to facilitate development of, and access to, innovative pharmaceutical products and processes.
LINK: https://www.ifpma.org/wp-content/uploads/2023/01/i2023_5.-Patent-Protection-as-a-Key-Driver-for-Pharmaceutical-Innovation.pdf


TITLE: Patent p

[3/18] Processing: Hygro Chemicals

TITLE: Hy-Gro Chemical Pharmtek Pvt. Ltd
SNIPPET: Hy-Gro operations comply with cGMP guide